# Saint-Ouen en mutation : recompositions socio-spatiales et dynamiques urbaines depuis les années 2010

*Projet Python pour la data science — Borhane Yahiaoui et Pauline Aurignac - 2026*

## Introduction

Commune limitrophe du nord de Paris en Seine-Saint-Denis, Saint-Ouen-sur-Seine a connu au cours des quinze dernières années une profonde transformation socio-économique. Longtemps associée à un passé industriel marqué par une forte présence ouvrière et une réputation de ville populaire en difficulté, elle s’impose aujourd’hui comme un pôle d’attractivité majeur de la première couronne parisienne. Portée par d’importants projets d’aménagement urbain, l’implantation de sièges sociaux et l’extension de la ligne 14 du métro, la commune attire désormais une population plus aisée et suscite un fort intérêt immobilier, au point d’incarner aux yeux de beaucoup un symbole de réhabilitation urbaine.
L’objectif de ce travail est d’examiner empiriquement à l’aide de Python plusieurs dimensions clés de l’évolution de Saint-Ouen depuis les années 2010. Nous étudions ainsi conjointement la dynamique démographique, l’évolution de la structure socio-économique, les transformations du parc de logements et du marché immobilier, ainsi que l’hétérogénéité des trajectoires entre les différents quartiers de la commune.

*Problématique : Dans quelle mesure les transformations récentes de Saint-Ouen s’accompagnent-elles d’une recomposition socio-spatiale, et comment celle-ci se différencie-t-elle selon les quartiers ?*

Pour y répondre, cette étude couvre la période allant du début des années 2010 aux données les plus récentes disponibles (notamment via le recensement de l’INSEE et les bases de transactions immobilières DVF). Nous étudions les évolutions à l’échelle de la commune et des quartiers, à l’aide du découpage infra-communal des IRIS (Îlots Regroupés pour l’Information Statistique).
Nous présentons dans un premier temps les sources mobilisées et les choix méthodologiques retenus pour harmoniser ces données dans le temps et dans l’espace. Nous analysons ensuite successivement les évolutions démographiques, socio-économiques et résidentielles de la commune, avant de croiser ces différentes dimensions afin de mettre en lumière les dynamiques communes et les contrastes persistants entre quartiers.

Cette approche se veut résolument descriptive : elle vise à mesurer et caractériser des évolutions ainsi que des corrélations spatiales et temporelles observées dans les données publiques, sans prétendre identifier à elle seule les mécanismes causaux à l’œuvre ni démontrer directement des phénomènes d’éviction résidentielle.


In [20]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from matplotlib.ticker import StrMethodFormatter
from pathlib import Path
import pyarrow.parquet as pq
import utils

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica", "DejaVu Sans", "Arial"]

**I- Récupération et exploration des bases de données principales**
A- Récupération des données 

1- Données géographiques

IRIS signifie « Ilots Regroupés pour l’Information Statistique ». L’Insee a créé ce découpage pour diffuser des statistiques à une échelle plus fine que celle de la commune. Un IRIS regroupe généralement autour de 2 000 habitants, mais sa taille peut varier. Ces unités permettent ici de comparer les quartiers de Saint-Ouen-sur-Seine.

Le « périmètre d’étude » désigne l’espace géographique retenu pour l’analyse : ici, les IRIS de la commune de Saint-Ouen-sur-Seine (code INSEE 93070). `zone_etude` est le GeoDataFrame qui contient les polygones de ces IRIS ; il sert à joindre les ventes aux quartiers et à cartographier les résultats. Ce périmètre n’est ni une zone tampon autour d’un transport ni une sélection de gares. La ligne 14 dispose d’un chargeur séparé dans `utils.py`, mais elle n’est pas utilisée dans l’analyse présentée ici.

In [22]:
gdf_iris = gpd.read_file("data/raw/IDF_iris.geojson")
zone_etude = utils.charger_zone_etude(code_insee="93070")

2- Données immobilières

Les données choisies pour observer les mutations immobilières sont les **données de valeurs foncières (DVF)**, dont la disponibilité est garantie par le décret n° 2018‑1350 du 28 décembre 2018 sur les cinq dernières anénes, avec des informations comme la date, la nature de la mutation, le type de local, sa valeur foncière, sa surface et ses coordonnées géographiques. Des données plus anciennes sont disponibles dans la rubrique "Ressources communautaires" de la page "Données de valeurs foncières" sur le site data.gouv.fr. Ces données ne sont pas fournies par les ministères économiques et financiers comme les données des 5 dernières années, mais elles n'étaient pas disponibles ailleurs pour des dates antérieures à 2021. Ainsi, les données de 2021 à 2026 mises à disposition sur data.gouv.fr ont été téléchargées au format .csv, et les données antérieures (de 2018 à 2020) au format .parquet. Les fichiers .parquet sont des fichiers nationaux tandis que les .csv sont limités à Saint-Ouen.

Remarque : aucune API gratuite n'est mise à disposition sur le site data.gouv.fr pour avoir accès aux données DVF. 

Le tableau ci-dessous montre quelques lignes des données de 2021 (fichier .csv) et de 2018 (fichier.parquet) tels qu’ils ont été fournis : 

In [ ]:
fichier_csv = Path("data/raw/93070.2021.csv")
fichier_parquet = Path("data/raw/dvf-2018.parquet")

df_csv = pd.read_csv(fichier_csv, nrows=5)

parquet = pq.ParquetFile(fichier_parquet)
df_parquet = next(parquet.iter_batches(batch_size=5)).to_pandas()

CSV — colonnes : ['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation', 'valeur_fonciere', 'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune', 'nom_commune', 'code_departement', 'ancien_code_commune', 'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle', 'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale', 'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude']


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2021-1592026,2021-05-31,1,Vente,3300000,6,NaN,RUE CHARLES SCHMIDT,1448,93400,...,Dépendance,NaN,0,S,sols,NaN,NaN,345,2.332088,48.905717
1,2021-1592026,2021-05-31,1,Vente,3300000,6,NaN,RUE CHARLES SCHMIDT,1448,93400,...,Dépendance,NaN,0,S,sols,NaN,NaN,345,2.332088,48.905717
2,2021-1592026,2021-05-31,1,Vente,3300000,6,NaN,RUE CHARLES SCHMIDT,1448,93400,...,Dépendance,NaN,0,S,sols,NaN,NaN,345,2.332088,48.905717
3,2021-1592026,2021-05-31,1,Vente,3300000,6,NaN,RUE CHARLES SCHMIDT,1448,93400,...,Dépendance,NaN,0,S,sols,NaN,NaN,345,2.332088,48.905717
4,2021-1592026,2021-05-31,1,Vente,3300000,6,NaN,RUE CHARLES SCHMIDT,1448,93400,...,Dépendance,NaN,0,S,sols,NaN,NaN,345,2.332088,48.905717


Parquet — colonnes : ['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation', 'valeur_fonciere', 'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune', 'nom_commune', 'code_departement', 'ancien_code_commune', 'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle', 'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale', 'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude']


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2018-1,2018-01-03,000001,Vente,109000.0,13.0,NaN,RUE GEN LOGEROT,1660,01000,...,Appartement,73.0,4.0,NaN,NaN,NaN,NaN,NaN,5.220440,46.200062
1,2018-1,2018-01-03,000001,Vente,109000.0,13.0,NaN,RUE GEN LOGEROT,1660,01000,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,5.220440,46.200062
2,2018-2,2018-01-04,000001,Vente,239300.0,4.0,NaN,RUE DE LA BARMETTE,0025,01250,...,Maison,163.0,4.0,S,sols,NaN,NaN,949.0,5.408041,46.255562
3,2018-2,2018-01-04,000001,Vente,239300.0,4.0,NaN,RUE DE LA BARMETTE,0025,01250,...,Maison,163.0,4.0,AG,terrains d'agrément,JARD,Jardin d'agrément,420.0,5.408041,46.255562
4,2018-2,2018-01-04,000001,Vente,239300.0,4.0,NaN,RUE DE LA BARMETTE,0025,01250,...,Maison,51.0,2.0,AG,terrains d'agrément,JARD,Jardin d'agrément,420.0,5.408041,46.255562


Pour travailler sur ces données on commence par les charger :

In [32]:
df_dvf_brut = utils.charger_dvf(
    code_insee=93070,
    dossier=None,
)

⚠️ 2017 : aucun fichier trouvé
✔ /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/raw/dvf-2018.parquet : 2296 lignes pour 2018
✔ /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/raw/dvf-2019.parquet : 3140 lignes pour 2019
✔ /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/raw/dvf-2020.parquet : 2540 lignes pour 2020
✔ /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/raw/93070.2021.csv : 3462 lignes pour 2021
✔ /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/raw/93070.2022.csv : 3019 lignes pour 2022
✔ /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/raw/93070.2023.csv : 3019 lignes pour 2023
✔ /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/raw/93070.2024.csv : 2471 lignes pour 2024
✔ /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/raw/93070.2025.csv : 2650 lignes pour 2025


Le helper `_aggreger_dvf_par_mutation`, appelé par `nettoyer_dvf`, transforme les lignes DVF en une observation par `id_mutation`. Il supprime d’abord les lignes strictement identiques, normalise les identifiants de mutation et écarte ceux qui sont absents. Pour chaque mutation, il :

- ne conserve que les ventes ou ventes en l’état futur d’achèvement comportant au moins un appartement ou une maison ;
- conserve une seule valeur foncière si une valeur non manquante unique est identifiable ; les mutations sans valeur exploitable ou avec plusieurs valeurs contradictoires sont écartées ;
- additionne les surfaces des seuls appartements et maisons ; une surface résidentielle manquante ou non positive entraîne l’exclusion de la mutation ;
- écarte les mutations dont la surface résidentielle totale est inférieure ou égale à 10 m² ou dont la valeur foncière est inférieure ou égale à 10 000 € ;
- calcule un prix au m² en divisant la valeur foncière de la mutation par la surface résidentielle totale, puis conserve les prix compris entre 2 000 et 15 000 €/m² ;
- exige une année cohérente et au moins une coordonnée résidentielle valide dans l’emprise géographique définie par le code.

Les seuils sont des choix de nettoyage : ils visent à écarter des observations peu plausibles pour notre analyse résidentielle, mais peuvent aussi exclure certaines transactions réelles. Nous les documentons et interprétons les résultats en conséquence.

Le prix au m² obtenu est un indicateur approximatif du prix résidentiel : la valeur foncière est celle de la mutation et peut inclure une dépendance vendue avec le logement. Les données DVF ne contiennent pas le code IRIS. Celui-ci est ajouté après ce nettoyage, par jointure spatiale entre les coordonnées des mutations et les polygones IRIS.

In [31]:
df_logements_nettoye = utils.nettoyer_dvf(df_dvf_brut)

Agrégation DVF : 1576 lignes strictement identiques supprimées; 2 mutations exclues pour valeur foncière absente/contradictoire; 0 pour nature absente/contradictoire; 0 pour surface résidentielle manquante/invalide; 84 sans coordonnée résidentielle valide; 934 avec années contradictoires; 0 avec dates contradictoires.
Filtre DVF terminé : 4576 mutations retenues sur 22597 lignes chargées.


Les fichiers DVF ne donnent pas directement le code IRIS de chaque mutation. Pour le construire, la fonction `joindre_ventes_aux_iris` du fichier utils exige des coordonnées `longitude` et `latitude` ainsi que le prix calculé `prix_m2`. Elle retire les observations sans coordonnées, crée un point pour chaque mutation en EPSG:4326, puis reprojette ces points dans le système de coordonnées des polygones IRIS si nécessaire.

La jointure spatiale utilise `predicate="within"` : un point est rattaché à l'IRIS dont le polygone le contient strictement. `how="left"` conserve toutes les mutations avec coordonnées; si aucune géométrie ne les contient, leur code IRIS reste manquant. Un point situé exactement sur la frontière d'un IRIS n'est pas considéré comme `within` et peut donc ne pas être rattaché. Ces cas sont conservés dans le tableau de jointure, mais exclus de l'agrégation par IRIS.

`agreger_dvf_par_iris` calcule ensuite, pour chaque IRIS, la médiane de `prix_m2` et le nombre d'observations. Comme le chargeur fournit désormais une ligne par `id_mutation`, `nb_ventes` compte ici les mutations retenues et rattachées, non les lots DVF d'origine. Le prix au m² reste un proxy : il divise la valeur totale de la mutation par la surface résidentielle totale et peut donc inclure la valeur d'une dépendance ou d'un autre bien vendu avec le logement.

In [ ]:
ventes_avec_iris = utils.joindre_ventes_aux_iris(
    df_logements_nettoye,
    zone_etude,
)

df_logements_final = pd.DataFrame(
    ventes_avec_iris.drop(columns="geometry")
)

Mutations jointes : 4576
Mutations sans IRIS : 0


Après le nettoyage des données DVF et leur jointure spatiale avec les polygones IRIS, nous enregistrons le tableau consolidé dans `data/processed/data_immo.parquet`. Il contient une ligne par mutation, les variables DVF retenues et le code IRIS correspondant. Les mutations qui n’ont pas pu être associées à un IRIS sont conservées, avec un code manquant. Nous pourrons ainsi réutiliser ces données dans la suite du notebook sans recharger les fichiers bruts ni refaire le nettoyage et la jointure.

In [ ]:
chemin_parquet_immo = utils.creer_data_immo_parquet(df_logements_final)
df_logements = pd.read_parquet(chemin_parquet_immo)

Parquet créé : /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/processed/data_immo.parquet (4576 lignes, années [2018, 2019, 2020, 2021, 2024, 2025]).
Parquet final : /Users/borhane/Documents/2A ENSAE/Python 2A/projet_python_2a_ensae/data/processed/data_immo.parquet


,id_mutation,date_mutation,annee,source_fichier,nature_mutation,type_local,types_locaux_residentiels,nb_lignes_residentielles,valeur_fonciere,surface_reelle_bati,prix_m2,longitude,latitude,code_commune,code_postal,nom_commune,code_iris
0,2018-1309994,2018-01-04,2018,dvf-2018.parquet,Vente en l'état futur d'achèvement,Appartement,Appartement,1,362900.0,66.0,5498.484848,2.346363,48.913999,93070,93400,Saint-Ouen-sur-Seine,930700117
1,2018-1309996,2018-01-11,2018,dvf-2018.parquet,Vente,Appartement,Appartement,1,382000.0,93.0,4107.526882,2.328267,48.902775,93070,93400,Saint-Ouen-sur-Seine,930700112
2,2018-1310001,2018-01-04,2018,dvf-2018.parquet,Vente,Appartement,Appartement,1,456000.0,116.0,3931.034483,2.341713,48.907472,93070,93400,Saint-Ouen-sur-Seine,930700118
3,2018-1310012,2018-01-05,2018,dvf-2018.parquet,Vente en l'état futur d'achèvement,Appartement,Appartement,1,306000.0,59.0,5186.440678,2.327871,48.909690,93070,93400,Saint-Ouen-sur-Seine,930700109
4,2018-1310013,2018-01-10,2018,dvf-2018.parquet,Vente en l'état futur d'achèvement,Appartement,Appartement,1,398500.0,65.0,6130.769231,2.335997,48.907520,93070,93400,Saint-Ouen-sur-Seine,930700106


**ATTENTION FAUT JUSTIFIER LES SEUILS BORHANE**

3- Données Filosofi

4- Données activités résidents 

5- Couples famille ménage 

6- Données logements